# 2 SQL DATABASE

**Project:** Irish Company Survival: Registrations, Cohort Survival and Prediction

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the clean company table into the SQL Server database `irish_company_survival`. It builds the dimension tables from the data, loads them before the fact table, and checks the row counts in SQL.

### Input
- `data/processed/companies_clean.csv` (from notebook 1)
- The empty tables created by `sql/00_create_database.sql` and `sql/01_create_tables.sql`

### Output
Six loaded tables in `irish_company_survival`: `dim_date`, `dim_company_type`, `dim_county`, `dim_sector`, `dim_status` and `fact_company`.

## Setup

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

## 1. Connect to the database

I connect to the local SQL Server with Windows Authentication and check that the connection reaches the project database.

In [2]:
# Connection to the local SQL Server database
connection_string = (
    "mssql+pyodbc://@localhost/irish_company_survival"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)
engine = create_engine(connection_string, fast_executemany=True)

# Check the connection
with engine.connect() as conn:
    database_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Connected to:", database_name)

Connected to: irish_company_survival


## 2. Load the clean company table

I load the clean table from notebook 1. The sector code is read as text so the leading zeros stay, and the two date columns are read as dates. Then I check the size, the missing values and the date range, because `dim_date` must cover every date in the table.

In [3]:
# Load the clean table from notebook 1
companies = pd.read_csv("../data/processed/companies_clean.csv",
                        dtype={"nace_code": "str"},
                        parse_dates=["reg_date", "closed_date"])

# Checks
print("Shape:", companies.shape,"\n")
print("Missing values:")
print(companies.isna().sum(), "\n")
print("Registration dates:", companies["reg_date"].min().date(), "to", companies["reg_date"].max().date())
print("Closing dates:     ", companies["closed_date"].min().date(), "to", companies["closed_date"].max().date())

Shape: (512008, 8) 

Missing values:
company_num          0
reg_date             0
company_type         0
status               0
closed               0
closed_date     305953
nace_code       246119
county            1822
dtype: int64 

Registration dates: 2000-01-01 to 2026-10-27
Closing dates:      2000-07-13 to 2026-10-21


The table has 512,008 companies and 8 columns, the same as the file saved in notebook 1. 305,953 companies have no closing date because they are still registered (512,008 minus the 206,055 closed). 246,119 have no sector code and 1,822 have no county. These three are the only columns with missing values, and they are the columns that may be NULL in the SQL tables. Registration dates run from 2000-01-01 to 2026-10-27, and every closing date falls inside this range.

## 3. Clear the tables

The SQL scripts create the empty tables. Here I delete any rows already in them, so the notebook gives the same result each time it runs. The fact table is cleared first, because its rows point to the dimension tables.

In [4]:
# Fact table first, then the dimension tables
tables = ["fact_company", "dim_date", "dim_company_type",
          "dim_county", "dim_sector", "dim_status"]

with engine.begin() as conn:
    for table in tables:
        conn.execute(text(f"DELETE FROM {table}"))

print("Cleared:", ", ".join(tables))

Cleared: fact_company, dim_date, dim_company_type, dim_county, dim_sector, dim_status


## 4. Load the dimension tables

### 4.1 Calendar (`dim_date`)

`dim_date` has one row per day, from the first date in the data to the last. I take both ends from the data, so the range is not typed by hand.

In [5]:
# First and last date across registration and closing dates
first_date = companies["reg_date"].min()
last_date = max(companies["reg_date"].max(), companies["closed_date"].max())

# One row per day
dim_date = pd.DataFrame({"date": pd.date_range(first_date, last_date, freq="D")})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["date"] = dim_date["date"].dt.date

# Load into SQL
dim_date.to_sql("dim_date", engine, if_exists="append", index=False)

# Check in SQL
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT COUNT(*), MIN([date]), MAX([date]), COUNT(DISTINCT [year]) FROM dim_date"
    )).one()

print("Rows in pandas:", len(dim_date))
print("Rows in SQL:   ", result[0])
print("Dates in SQL:  ", result[1], "to", result[2])
print("Years in SQL:  ", result[3])

Rows in pandas: 9797
Rows in SQL:    9797
Dates in SQL:   2000-01-01 to 2026-10-27
Years in SQL:   27


`dim_date` has 9,797 days, from 2000-01-01 to 2026-10-27, covering 27 years. The row count in SQL matches the table built in pandas.

### 4.2 Company types (`dim_company_type`)

I take the company types from the data in alphabetical order and give each one an ID from 1.

In [6]:
# One row per company type, IDs from 1
company_types = sorted(companies["company_type"].unique())
dim_company_type = pd.DataFrame({
    "company_type_id": range(1, len(company_types) + 1),
    "company_type": company_types
})

# Load into SQL
dim_company_type.to_sql("dim_company_type", engine, if_exists="append", index=False)

# Check in SQL
with engine.connect() as conn:
    sql_rows = conn.execute(text("SELECT COUNT(*) FROM dim_company_type")).scalar()

print("Rows in pandas:", len(dim_company_type))
print("Rows in SQL:   ", sql_rows)
dim_company_type

Rows in pandas: 7
Rows in SQL:    7


,company_type_id,company_type
0,1,DAC
1,2,Guarantee (CLG)
2,3,Other
3,4,Private limited (LTD)
4,5,Public (PLC)
5,6,Unknown
6,7,Unlimited


`dim_company_type` has the 7 company types from notebook 1, with IDs 1 to 7. The row count in SQL matches.

### 4.3 Counties (`dim_county`)

I take the counties found in the address in notebook 1. Companies with no county get no ID, so the missing values are left out here.

In [7]:
# One row per county, IDs from 1 (missing counties left out)
counties = sorted(companies["county"].dropna().unique())
dim_county = pd.DataFrame({
    "county_id": range(1, len(counties) + 1),
    "county": counties
})

# Load into SQL
dim_county.to_sql("dim_county", engine, if_exists="append", index=False)

# Check in SQL
with engine.connect() as conn:
    sql_rows = conn.execute(text("SELECT COUNT(*) FROM dim_county")).scalar()

print("Rows in pandas:", len(dim_county))
print("Rows in SQL:   ", sql_rows)

Rows in pandas: 26
Rows in SQL:    26


`dim_county` has all 26 counties, with IDs 1 to 26. The row count in SQL matches. The 1,822 companies with no county get no county ID in the fact table.

### 4.4 Sectors (`dim_sector`)

The first two digits of a sector code are its division, and each range of divisions belongs to one NACE Rev 2 section. For example, codes starting with 41 to 43 are section F, Construction. I type the section ranges and short section names from the NACE Rev 2 standard, because the data does not contain them.

In [8]:
# NACE Rev 2: last division in each section (A = 01 to 03, B = 05 to 09, ...)
last_division = [3, 9, 33, 35, 39, 43, 47, 53, 56, 63, 66,
                 68, 75, 82, 84, 85, 88, 93, 96, 98, 99]
section_letters = list("ABCDEFGHIJKLMNOPQRSTU")
section_names = [
    "Agriculture", "Mining", "Manufacturing", "Energy", "Water and waste",
    "Construction", "Wholesale and retail", "Transport", "Accommodation and food",
    "Information and communication", "Finance and insurance", "Real estate",
    "Professional services", "Administrative services", "Public administration",
    "Education", "Health and social work", "Arts and recreation",
    "Other services", "Households", "Extraterritorial bodies"
]

In [9]:
# One row per sector code, IDs from 1 (missing codes left out)
nace_codes = sorted(companies["nace_code"].dropna().unique())
dim_sector = pd.DataFrame({
    "sector_id": range(1, len(nace_codes) + 1),
    "nace_code": nace_codes
})

# Section from the first two digits of the code
division = dim_sector["nace_code"].str[:2].astype(int)
bins = [0] + last_division
dim_sector["nace_section"] = pd.cut(division, bins=bins, labels=section_letters)
dim_sector["section_name"] = pd.cut(division, bins=bins, labels=section_names)

dim_sector.head(3)

,sector_id,nace_code,nace_section,section_name
0,1,0111,A,Agriculture
1,2,0113,A,Agriculture
2,3,0115,A,Agriculture


In [10]:
# Load into SQL
dim_sector.to_sql("dim_sector", engine, if_exists="append", index=False)

# Check in SQL
with engine.connect() as conn:
    sql_rows = conn.execute(text("SELECT COUNT(*) FROM dim_sector")).scalar()

print("Rows in pandas:", len(dim_sector))
print("Rows in SQL:   ", sql_rows)
print("Sections:", dim_sector["nace_section"].nunique())

Rows in pandas: 581
Rows in SQL:    581
Sections: 21


`dim_sector` has 581 sector codes, with IDs 1 to 581, and all 21 NACE Rev 2 sections appear. The row count in SQL matches. The 246,119 companies with no sector code get no sector ID in the fact table.

### 4.5 Statuses (`dim_status`)

Each company has one CRO status. `closed` comes from notebook 1 and is True when the company has left the register. `closed_strict` also counts `Strike Off Listed` and `Liquidation` as closed, because those companies are on their way off the register. I use it later as a sensitivity check.

In [11]:
# One row per status, with its closed flag from notebook 1
dim_status = companies[["status", "closed"]].drop_duplicates().sort_values("status")
dim_status["status_id"] = range(1, len(dim_status) + 1)

# Strict rule: also count companies on their way off the register as closed
dim_status["closed_strict"] = (dim_status["closed"]
                               | dim_status["status"].isin(["Strike Off Listed", "Liquidation"]))

# Check: each status has only one closed value
print("Rows:", len(dim_status))
print("Different statuses:", companies["status"].nunique())
dim_status[["status_id", "status", "closed", "closed_strict"]]

Rows: 20
Different statuses: 20


,status_id,status,closed,closed_strict
512007,1,Administration,False,False
2256,2,Administration (UK),False,False
10219,3,Ceased,True,True
508184,4,Ceased IRL,True,True
36839,5,Conv. to IndProvSoc,True,True
26002,6,De-Registered ICAV,True,True
201964,7,De-Registered Investment Company,True,True
11618,8,Deleted CB Conversion,True,True
1577,9,Deleted CB Merger,True,True
17909,10,Deleted as per SE Tr,True,True


There are 20 statuses, and each one has only one closed value. 14 statuses count as closed. The strict rule adds `Liquidation` and `Strike Off Listed`, so 16 count as closed under it.

In [12]:
# Load into SQL
dim_status.to_sql("dim_status", engine, if_exists="append", index=False)

# Check in SQL
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT COUNT(*), SUM(CAST(closed AS INT)), SUM(CAST(closed_strict AS INT)) FROM dim_status"
    )).one()

print("Rows in SQL:          ", result[0])
print("Closed statuses:      ", result[1])
print("Strict closed statuses:", result[2])

Rows in SQL:           20
Closed statuses:       14
Strict closed statuses: 16


`dim_status` has 20 statuses in SQL. 14 are closed and 16 are closed under the strict rule, the same as in pandas.

## 5. Load the fact table

### 5.1 Build `fact_company`

The fact table has one row per company. Instead of names, it stores the IDs from the dimension tables. I look up each company's type, status, county and sector code in its dimension table to get the ID. Companies with no county or no sector code keep a missing ID.

In [15]:
# Lookups from name to ID
type_ids = dim_company_type.set_index("company_type")["company_type_id"]
status_ids = dim_status.set_index("status")["status_id"]
county_ids = dim_county.set_index("county")["county_id"]
sector_ids = dim_sector.set_index("nace_code")["sector_id"]

# One row per company, with IDs instead of names
fact_company = pd.DataFrame({
    "company_num": companies["company_num"],
    "reg_date": companies["reg_date"].dt.date,
    "closed_date": companies["closed_date"].dt.date,
    "company_type_id": companies["company_type"].map(type_ids),
    "status_id": companies["status"].map(status_ids),
    "county_id": companies["county"].map(county_ids).astype("Int64"),
    "sector_id": companies["nace_code"].map(sector_ids).astype("Int64"),
})

# Check: only the expected columns have missing values
print("Shape:", fact_company.shape, "\n")
print(fact_company.isna().sum())
fact_company.head(3)

Shape: (512008, 7) 

company_num             0
reg_date                0
closed_date        305953
company_type_id         0
status_id               0
county_id            1822
sector_id          246119
dtype: int64


,company_num,reg_date,closed_date,company_type_id,status_id,county_id,sector_id
0,500,2000-01-01,NaT,3,18,<NA>,<NA>
1,316262,2000-01-04,NaT,4,18,6,<NA>
2,317562,2000-01-06,NaT,2,18,6,<NA>


The fact table has 512,008 companies and 7 columns. Every company found its company type ID and status ID. The missing values are the same as in the clean table: 305,953 closing dates, 1,822 county IDs and 246,119 sector IDs.

### 5.2 Load `fact_company` and check it in SQL

I load the 512,008 companies into SQL. The foreign keys reject any row whose ID is not in a dimension table, so a successful load already shows that every ID links. Then I check the totals in SQL against the clean table.

In [16]:
# Load into SQL
fact_company.to_sql("fact_company", engine, if_exists="append", index=False)

# Check in SQL: totals, and closed companies through the link to dim_status
check_query = """
SELECT COUNT(*)                     AS companies,
       COUNT(f.closed_date)         AS with_closed_date,
       COUNT(f.county_id)           AS with_county,
       COUNT(f.sector_id)           AS with_sector,
       SUM(CAST(s.closed AS INT))   AS closed
FROM fact_company AS f
JOIN dim_status AS s
    ON f.status_id = s.status_id
"""
sql_check = pd.read_sql(text(check_query), engine)

print("Companies in pandas:", len(fact_company))
print("Closed in pandas:   ", companies["closed"].sum())
sql_check

Companies in pandas: 512008
Closed in pandas:    206055


,companies,with_closed_date,with_county,with_sector,closed
0,512008,206055,510186,265889,206055


All 512,008 companies loaded into SQL, so every ID links to its dimension table. In SQL, 206,055 companies have a closing date and 206,055 are closed through `dim_status`, the same as in the clean table. 510,186 companies have a county and 265,889 have a sector code.

## 6. Summary

1. **The database has six tables in a star schema.** `fact_company` has one row per company, and five dimension tables hold the calendar, company types, counties, sectors and statuses.
2. **All 512,008 companies loaded.** The foreign keys accepted every row, so every ID in the fact table links to its dimension table.
3. **The SQL totals match the clean table.** 206,055 companies are closed, and the same 206,055 have a closing date.
4. **Missing values are kept as NULL.** 1,822 companies have no county and 246,119 have no sector code. They stay in the fact table with an empty ID.
5. **Two closed rules are stored in `dim_status`.** `closed` counts 14 statuses as closed. `closed_strict` also counts `Liquidation` and `Strike Off Listed`, for a sensitivity check later.

**Limitations**
- The sector sections are worked out from the first two digits of the code. All 581 codes in the data fall in a NACE Rev 2 division, but the check is at division level, not for every 4-digit code.
- The `Unknown` company type has only 2 companies, too few for its own survival rate.